In [20]:
import os
import csv
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import video as models_3d
import torchvision.transforms as transforms

##############################################################################
#                              CONFIGURATION                                 #
##############################################################################

CSV_PATH = "dataset_split.csv"    # CSV with columns: filepath,label,split
TRAIN_SPLIT = "train"
VAL_SPLIT   = "val"
TEST_SPLIT  = "test"

# Number of frames in each video (.npy)
FRAMES_PER_VIDEO = 120            # or however many you have

# Desired spatial size after resize (typical for 3D ResNet is 112×112)
RESIZE_HEIGHT = 112
RESIZE_WIDTH  = 112

# Training parameters
BATCH_SIZE    = 2
EPOCHS        = 10
LEARNING_RATE = 1e-4

# Decide if you want to do any data augmentation
# (e.g., random crop, random horizontal flip). For simplicity, we’ll just resize.
# You can add more transforms if needed.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:


##############################################################################
#                           DATASET DEFINITION                               #
##############################################################################

class Video3DDataset(Dataset):
    """
    Loads a .npy video of shape (T, H, W, 3).
    Resizes each frame to (RESIZE_HEIGHT, RESIZE_WIDTH).
    Converts to shape (3, T, RESIZE_HEIGHT, RESIZE_WIDTH) for 3D CNN input.
    Returns a (C, T, H, W) tensor and an integer label.
    """
    def __init__(self, csv_file, split="train"):
        super().__init__()
        
        self.samples = []
        
        # 1. Read CSV, filter rows by 'split'
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    filepath = row["filepath"]
                    label    = row["label"]
                    self.samples.append((filepath, label))
        
        # 2. Build label → index mapping
        unique_labels = sorted(set([s[1] for s in self.samples]))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        
        # 3. Convert each label to an integer
        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]
        
        # 4. Define transforms (spatial only). 
        #    For 3D CNN, time-based augmentation is trickier. 
        #    You can add random spatial transforms here if you like.
        self.resize_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((RESIZE_HEIGHT, RESIZE_WIDTH)),  # (height, width)
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, label_idx = self.samples[idx]
        frames = np.load(filepath)  # shape (T, H, W, 3)
        
        # Optional: if the array is uint8 [0..255], you might want to convert to float [0..1]
        # frames = frames.astype(np.float32) / 255.0
        
        # We'll store transformed frames here
        processed_frames = []
        
        # 1. For each frame in the (T, H, W, 3)
        for frame in frames:
            # frame shape = (H, W, 3), likely uint8
            
            # 2. Convert to PIL, resize
            pil_img = self.resize_transform(frame)  # resize to (112, 112)
            
            # 3. Convert back to numpy array if needed
            #    Or directly to a Tensor. Here, let's do numpy first for clarity:
            resized_np = np.array(pil_img)  # shape (112, 112, 3)
            
            # 4. Append to list
            processed_frames.append(resized_np)
        
        # Now processed_frames is a list of length T, each (112, 112, 3)
        processed_frames = np.stack(processed_frames, axis=0)  # shape (T, 112, 112, 3)
        
        # 5. Transpose to (3, T, 112, 112)
        processed_frames = processed_frames.transpose((3, 0, 1, 2))
        
        # Convert to torch float tensor
        video_tensor = torch.from_numpy(processed_frames).float()  # shape (3, T, 112, 112)
        
        # (Optional) If you want to normalize using something like Kinetics-400 means/stdevs:
        # means = [0.43216, 0.394666, 0.37645]  # example for Kinetics
        # stds  = [0.22803, 0.22145, 0.216989]
        # for c in range(3):
        #     video_tensor[c] = (video_tensor[c] / 255.0 - means[c]) / stds[c]
        
        return video_tensor, label_idx



In [22]:

##############################################################################
#                       3D RESNET MODEL DEFINITION                           #
##############################################################################

def create_3d_resnet(num_classes):
    """
    Creates an r3d_18 model from torchvision, pretrained on Kinetics-400 (if available).
    We replace the final layer to match num_classes.
    """
    # 1. Load a pretrained 3D ResNet-18
    model_3d = models_3d.r3d_18(pretrained=True)
    
    # 2. The final fully connected layer is model_3d.fc
    in_feats = model_3d.fc.in_features  # typically 512
    model_3d.fc = nn.Linear(in_feats, num_classes)
    
    return model_3d


In [27]:


##############################################################################
#                         TRAINING & EVALUATION LOOP                         #
##############################################################################

def train_3d_cnn():
    # 1. Create datasets
    train_dataset = Video3DDataset(CSV_PATH, split=TRAIN_SPLIT)
    val_dataset   = Video3DDataset(CSV_PATH, split=VAL_SPLIT)
    test_dataset  = Video3DDataset(CSV_PATH, split=TEST_SPLIT)
    
    # 2. DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3. Number of classes
    num_classes = len(train_dataset.label_to_idx)
    
    # 4. Create model, loss, optimizer
    model = create_3d_resnet(num_classes)
    model.to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 5. Training loop
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for batch_idx, (videos, labels) in enumerate(train_loader):
            # videos shape: (B, 3, T=120, 112, 112)
            # labels shape: (B,)
            videos = videos.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(videos)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        
        # 6. Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for videos, labels in val_loader:
                videos = videos.to(DEVICE)
                labels = labels.to(DEVICE)
                
                outputs = model(videos)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, preds = torch.max(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_acc = correct / total if total > 0 else 0
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_acc*100:.2f}%")
    
    # 7. Final Test
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    
    with torch.no_grad():
        for videos, labels in test_loader:
            videos = videos.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            _, preds = torch.max(outputs, dim=1)
            correct_test += (preds == labels).sum().item()
            total_test += labels.size(0)
    
    avg_test_loss = test_loss / len(test_loader) if len(test_loader) > 0 else 0
    test_acc = correct_test / total_test if total_test > 0 else 0
    
    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_acc*100:.2f}%")

    # Save the trained model
    torch.save(model.state_dict(), "3d_cnn_model.pth")
    print("Model saved.")
##############################################################################
#                           MAIN ENTRY POINT                                 #
##############################################################################



In [ ]:
if __name__ == "__main__":
    train_3d_cnn()


/Users/berkebozaci/.pyenv/versions/3.11.3/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/berkebozaci/.pyenv/versions/3.11.3/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/10] | Train Loss: 0.7902 | Val Loss: 0.0574 | Val Acc: 100.00%


In [ ]:
# Recreate the model
model = create_3d_resnet(num_classes=3)  # Replace 3 with the actual number of classes
model.load_state_dict(torch.load("3d_cnn_model.pth"))
model.to(DEVICE)
model.eval()
print("Model loaded and ready for testing.")


In [ ]:
def predict_video(video_npy_path, model, label_map, device='cpu'):
    """
    Predict the label for a single video using the trained model.
    """
    frames = np.load(video_npy_path)  # shape (T, H, W, 3)
    
    # Preprocess frames
    processed_frames = []
    for frame in frames:
        frame = cv2.resize(frame, (RESIZE_WIDTH, RESIZE_HEIGHT))  # Resize to model's input size
        processed_frames.append(frame)
    processed_frames = np.stack(processed_frames, axis=0).transpose((3, 0, 1, 2))  # (T, H, W, C) -> (C, T, H, W)
    video_tensor = torch.from_numpy(processed_frames).float().unsqueeze(0).to(device)  # Add batch dim
    
    # Predict
    with torch.no_grad():
        outputs = model(video_tensor)
        _, pred_idx = torch.max(outputs, dim=1)
        pred_label = label_map[pred_idx.item()]
    
    return pred_label


In [ ]:
import csv

# Function to recreate label_to_idx
def load_label_to_idx(csv_file):
    """
    Reads the CSV and creates the label_to_idx mapping.
    """
    with open(csv_file, "r") as f:
        reader = csv.DictReader(f)
        labels = set(row["label"] for row in reader)
    label_to_idx = {lbl: idx for idx, lbl in enumerate(sorted(labels))}
    return label_to_idx

# Load the mapping
label_to_idx = load_label_to_idx(CSV_PATH)

# Reverse the mapping to get idx_to_label
idx_to_label = {idx: lbl for lbl, idx in label_to_idx.items()}

# Path to a sample video
# Test your video
video_path = "processed_frames_test/class/class-test.npy"  # Replace with your test file path
#video_path = "processed_frames_test/room/room-test.npy"  # for example
#video_path = "processed_frames_test/teach/teach-test.npy"  # for example

# Predict the label
predicted_label = predict_video(video_path, model, idx_to_label, device=DEVICE)
print(f"Predicted Label: {predicted_label}")
